# MIMIC-OMOP Analytics con Spark SQL e Iceberg

## Objetivo
Conectar Apache Spark a MinIO, leer tablas Iceberg de la capa Silver (omop-silver) y ejecutar consultas anala­ticas sobre datos cli­nicos de pacientes y diagnosticos.

## Paso 1: Instalación de dependencias

In [1]:
import subprocess, sys, os

# JARs se guardan en el volumen montado para no descargarlos cada vez
JAR_DIR = "/home/jovyan/work/jars"
os.makedirs(JAR_DIR, exist_ok=True)

MAVEN = "https://repo1.maven.org/maven2"
JARS = [
    (f"{MAVEN}/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/"
     "iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
     "iceberg-spark-runtime-3.5_2.12-1.6.1.jar"),
    (f"{MAVEN}/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar",
     "hadoop-aws-3.3.4.jar"),
    (f"{MAVEN}/com/amazonaws/aws-java-sdk-bundle/1.12.262/"
     "aws-java-sdk-bundle-1.12.262.jar",
     "aws-java-sdk-bundle-1.12.262.jar"),
]

for url, filename in JARS:
    path = f"{JAR_DIR}/{filename}"
    if os.path.exists(path):
        print(f"Ya existe: {filename}")
    else:
        print(f"Descargando {filename} (puede tardar)...")
        subprocess.run(["wget", "-q", "--show-progress", "-O", path, url], check=True)
        print(f"OK: {filename}")

subprocess.check_call([sys.executable, "-m", "pip", "install", "pyiceberg", "-q"])
print("\nTodos los JARs y dependencias listos.")

Ya existe: iceberg-spark-runtime-3.5_2.12-1.6.1.jar
Ya existe: hadoop-aws-3.3.4.jar
Ya existe: aws-java-sdk-bundle-1.12.262.jar

Todos los JARs y dependencias listos.


## Paso 2: Inicializar Spark con Iceberg y MinIO

In [ ]:
from pyspark.sql import SparkSession
import os

existing = SparkSession.getActiveSession()
if existing:
    existing.stop()

JAR_DIR = "/home/jovyan/work/jars"
JARS = ",".join([
    f"{JAR_DIR}/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
    f"{JAR_DIR}/hadoop-aws-3.3.4.jar",
    f"{JAR_DIR}/aws-java-sdk-bundle-1.12.262.jar",
])

MINIO_ACCESS = os.environ["MINIO_ROOT_USER"]
MINIO_SECRET = os.environ["MINIO_ROOT_PASSWORD"]

spark = SparkSession.builder \
    .appName("MIMIC-OMOP-Analytics") \
    .config("spark.jars", JARS) \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hive") \
    .config("spark.sql.catalog.iceberg.uri", "thrift://hive-metastore:9083") \
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://omop-silver") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3.access.key", MINIO_ACCESS) \
    .config("spark.hadoop.fs.s3.secret.key", MINIO_SECRET) \
    .config("spark.hadoop.fs.s3.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3.path.style.access", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} listo | Iceberg 1.6.1")

## Paso 3: Leer tablas Iceberg de la capa Silver

In [ ]:
# Tablas disponibles en el catalogo
spark.sql("SHOW TABLES IN iceberg.omop").show(truncate=False)

# Cargar tabla person
df_person = spark.read.table("iceberg.omop.person")
print("=== person ===")
df_person.printSchema()
df_person.show(5, truncate=False)

# Cargar tabla condition_occurrence
df_conditions = spark.read.table("iceberg.omop.condition_occurrence")
print("=== condition_occurrence ===")
df_conditions.printSchema()
df_conditions.show(5, truncate=False)

## Consulta 1: Top 10 diagnósticos más frecuentes por género

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_joined = df_conditions.join(df_person, on="person_id", how="inner")

ventana_genero = Window.partitionBy("gender_source_value").orderBy(F.desc("total"))

top_diagnosticos_por_genero = (
    df_joined
    .groupBy("gender_source_value", "condition_source_value")
    .agg(F.count("*").alias("total"))
    .withColumn("rank", F.rank().over(ventana_genero))
    .filter(F.col("rank") <= 10)
    .orderBy("gender_source_value", "rank")
)

print("=== Top 10 diagnósticos (código ICD) por género ===")
top_diagnosticos_por_genero.show(20, truncate=False)

=== Top 10 diagnósticos (código ICD) por género ===
+-------------------+----------------------+-----+----+
|gender_source_value|condition_source_value|total|rank|
+-------------------+----------------------+-----+----+
|F                  |4019                  |18   |1   |
|F                  |3051                  |13   |2   |
|F                  |42731                 |12   |3   |
|F                  |4241                  |12   |3   |
|F                  |5990                  |7    |5   |
|F                  |V1204                 |6    |6   |
|F                  |V5861                 |6    |6   |
|F                  |71590                 |6    |6   |
|F                  |4293                  |6    |6   |
|F                  |E8497                 |6    |6   |
|F                  |30500                 |6    |6   |
|F                  |78062                 |6    |6   |
|F                  |5262                  |6    |6   |
|F                  |4271                  |6    |6 

## Consulta 2: Comorbilidades más frecuentes por visita hospitalaria

In [5]:
c1 = df_conditions.alias("c1")
c2 = df_conditions.alias("c2")

comorbilidades = (
    c1.join(
        c2,
        on=(F.col("c1.visit_occurrence_id") == F.col("c2.visit_occurrence_id")) &
           (F.col("c1.condition_source_value") < F.col("c2.condition_source_value")),
        how="inner"
    )
    .groupBy(
        F.col("c1.condition_source_value").alias("diagnostico_1"),
        F.col("c2.condition_source_value").alias("diagnostico_2")
    )
    .agg(F.count("*").alias("co_ocurrencias"))
    .orderBy(F.desc("co_ocurrencias"))
)

print("=== Top 10 pares de diagnósticos que más co-ocurren en la misma visita ===")
comorbilidades.show(10, truncate=False)

=== Top 10 pares de diagnósticos que más co-ocurren en la misma visita ===
+-------------+-------------+--------------+
|diagnostico_1|diagnostico_2|co_ocurrencias|
+-------------+-------------+--------------+
|2724         |4019         |4             |
|4019         |41401        |3             |
|4019         |42731        |3             |
|99859        |E8782        |3             |
|4019         |4241         |3             |
|2724         |41401        |3             |
|4241         |42731        |3             |
|04185        |99859        |2             |
|I4891        |Y92230       |2             |
|56722        |V453         |2             |
+-------------+-------------+--------------+
only showing top 10 rows

